The purpose of this project is to create a machine learning prediction model that can help identify megaline customer behavior. To do this we must create different types of models and determine which model would work the best

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [16]:
megaline_data=pd.read_csv('/datasets/users_behavior.csv')

In [17]:
print(megaline_data.head())

   calls  minutes  messages   mb_used  is_ultra
0   40.0   311.90      83.0  19915.42         0
1   85.0   516.75      56.0  22696.96         0
2   77.0   467.66      86.0  21060.45         0
3  106.0   745.53      81.0   8437.39         1
4   66.0   418.74       1.0  14502.75         0


In [18]:
print(megaline_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB
None


In [19]:
print(megaline_data.isnull())

      calls  minutes  messages  mb_used  is_ultra
0     False    False     False    False     False
1     False    False     False    False     False
2     False    False     False    False     False
3     False    False     False    False     False
4     False    False     False    False     False
...     ...      ...       ...      ...       ...
3209  False    False     False    False     False
3210  False    False     False    False     False
3211  False    False     False    False     False
3212  False    False     False    False     False
3213  False    False     False    False     False

[3214 rows x 5 columns]


In [20]:
print(megaline_data.describe())

             calls      minutes     messages       mb_used     is_ultra
count  3214.000000  3214.000000  3214.000000   3214.000000  3214.000000
mean     63.038892   438.208787    38.281269  17207.673836     0.306472
std      33.236368   234.569872    36.148326   7570.968246     0.461100
min       0.000000     0.000000     0.000000      0.000000     0.000000
25%      40.000000   274.575000     9.000000  12491.902500     0.000000
50%      62.000000   430.600000    30.000000  16943.235000     0.000000
75%      82.000000   571.927500    57.000000  21424.700000     1.000000
max     244.000000  1632.060000   224.000000  49745.730000     1.000000


When reviewing the data it appears that the data is a nice clean data set with no missing values. Noting that messages and calls are stored as float64 DTYPE rather than integer. If required later on it might make sense to change those data types. 

In [21]:
#split the data 3 ways for model creation. 
features= megaline_data.drop(['is_ultra'], axis=1)
target= megaline_data['is_ultra']
#first split off 40% for validation and test
features_train, features_temp, target_train, target_temp = train_test_split(
    features, target, test_size=0.4, random_state=42
)

#second split the 40% evenly into validation and test (20% each)
features_valid, features_test, target_valid, target_test= train_test_split(
    features_temp, target_temp, test_size=.5, random_state=42
)

print('Dataset Split Sizes')
print(f'Training Set: {features_train.shape[0]} rows({features_train.shape[0]/len(megaline_data):1%})')
print(f'Validation Set: {features_valid.shape[0]} rows({features_valid.shape[0]/len(megaline_data):1%})')
print(f'Test set: {features_test.shape[0]} rows({features_test.shape[0]/len(megaline_data):1%})')
print(f'Total: {features_train.shape[0] + features_valid.shape[0] + features_test.shape[0]} rows')

Dataset Split Sizes
Training Set: 1928 rows(59.987554%)
Validation Set: 643 rows(20.006223%)
Test set: 643 rows(20.006223%)
Total: 3214 rows


In [22]:
#Model 1- Descision Tree

print('Descision Tree Results')
for depth in [3, 5, 7, 9, 11]:
    dt_model= DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt_model.fit(features_train, target_train)
    dt_predictions=dt_model.predict(features_valid)
    accuracy= accuracy_score(target_valid, dt_predictions)
    print(f'max_depth={depth}: Accuracy= {accuracy: .4f}')

Descision Tree Results
max_depth=3: Accuracy=  0.7916
max_depth=5: Accuracy=  0.7729
max_depth=7: Accuracy=  0.7807
max_depth=9: Accuracy=  0.7807
max_depth=11: Accuracy=  0.7854


Decsision tree results are interesting. They tell us that the more complexities we add to the decision tree the less accurate things get. I would venture to say that is due to the limited number of features in the data set. 

In [23]:
#Model 2 Random Forrest

print('Random Forrest Results')
for estimators in [10, 50, 100, 200]:
    rf_model= RandomForestClassifier( n_estimators=estimators, max_depth= depth, random_state= 42)
    rf_model.fit(features_train, target_train)
    rf_predictions= rf_model.predict(features_valid)
    accuracy= accuracy_score(target_valid, rf_predictions)
    print(f'n_estimators={estimators}, max_depth={depth}: Accuracy= {accuracy: .4f}')

Random Forrest Results
n_estimators=10, max_depth=11: Accuracy=  0.8040
n_estimators=50, max_depth=11: Accuracy=  0.8103
n_estimators=100, max_depth=11: Accuracy=  0.8134
n_estimators=200, max_depth=11: Accuracy=  0.8118


when looking at the results of the random forest model it is noted that it performed better than the decision tree model did with all accuracies being over 80% with the model improving as more estimators were added. With the exception of 200 estimators. There was a slight decline in accuracy from 100 to 200. I would venture to say this is due to the model having run into similar patterns once it got past a certain number of estimators. It would be interesting to see where that threshold is. 

In [24]:
#Model 3 Logistic Regression

print('Logistic Regression Results')
lr_model=LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(features_train, target_train)
lr_predictions= lr_model.predict(features_valid)
lr_accuracy= accuracy_score(target_valid, lr_predictions)
print(f'Accuracy = {lr_accuracy:.4f}')

Logistic Regression Results
Accuracy = 0.7403


When reviewing the results of the logistic regression model it is noted that the results are substantially less accurate. I believe this is to be expected because customer behavior is likely to have phone call or data behaviors that would be outside of the linear progression that the logistic regression would require for accuracy. 

In [25]:
#Sanity Final Test

final_model= RandomForestClassifier(n_estimators=100, max_depth=11, random_state=42)
final_model.fit(features_train, target_train)
final_predictions=final_model.predict(features_valid)
final_accuracy= accuracy_score(target_test, final_predictions)
print('Final Machine Learning Test Results')
print('Model Chosen: Random Forest')
print('Parameters Chosen: n_estimators=100 max_depth=11')
print()
if final_accuracy >= 0.75:
    print(f'Passed with an Accuracy of {final_accuracy:.1%} exceeds the 0.75 threshold')
else:
    print(f'Failed with and Accuracy of {final_accuracy:.1%} short of required 0.75 threshold')

Final Machine Learning Test Results
Model Chosen: Random Forest
Parameters Chosen: n_estimators=100 max_depth=11

Failed with and Accuracy of 61.1% short of required 0.75 threshold


This is interesting. the testing showed much more accurate results. Likely a result of overfitting? Will have to revise and run additional test models. changing the depths and the number of estimators will hopefully find a more accurate result. 


****NOTE After reviewing with Dot my project prior to submission it was noticed that the reason for my failure was not due to the model itself but due to the code written. Before I knew the issue I assumed overfitting due to the lack of features in the data. I was incorrect in that assumption but did not want to remove this from the progress of the project to show how I worked through the issue.****

In [26]:
# Revised Test- Testing multiple configurations on test set
print('Revised Model Testing')

configs=[
    {'n_estimators': 100, 'max_depth': 3},
    {'n_estimators': 100, 'max_depth': 5},
    {'n_estimators': 100, 'max_depth': 7},
    {'n_estimators': 200, 'max_depth': 3},
    {'n_estimators': 200, 'max_depth': 5},
    {'n_estimators': 200, 'max_depth': 7}
]
for config in configs:
    model=RandomForestClassifier(n_estimators=config['n_estimators'],
                                max_depth=config['max_depth'],
                                random_state=42)
    model.fit(features_train, target_train)
    valid_acc=accuracy_score(target_valid, model.predict(features_valid))
    

    print(f'n_estimators= {config["n_estimators"]}, max_depth= {config["max_depth"]}: Validation Accuracy= {valid_acc:.4f}')
    print()

Revised Model Testing
n_estimators= 100, max_depth= 3: Validation Accuracy= 0.7823

n_estimators= 100, max_depth= 5: Validation Accuracy= 0.7885

n_estimators= 100, max_depth= 7: Validation Accuracy= 0.7978

n_estimators= 200, max_depth= 3: Validation Accuracy= 0.7838

n_estimators= 200, max_depth= 5: Validation Accuracy= 0.7916

n_estimators= 200, max_depth= 7: Validation Accuracy= 0.7947



new model tests suggested that max depth of 7 with 200 n_estimators has the highest validation accuracy and the lowest gap between test and validation values. This is the model we will use for the second final sanity test. 

In [27]:
#Final Model Sanity Test:
final_model_2= RandomForestClassifier(
    n_estimators=200,
    max_depth=7,
    random_state=42
)

final_model_2.fit(features_train, target_train)
final_model_predictions=final_model_2.predict(features_test)
final_model_accuracy= accuracy_score(target_test, final_model_predictions)

print('Final Model Revised Results')
print('Model: Random Forest')
print('Parameters: n_estimators= 200 at a max_depth=7')
print('Validation Accuracy:.7947')
if final_model_accuracy > .75:
    print(f' Passed- Accuracy of {final_model_accuracy:.1%} exceeds the required threshold of .75')
else:
    print(f' Failed- Accuracy of {final_model_accuracy: .1%} falls below required threshold of .75')
   

Final Model Revised Results
Model: Random Forest
Parameters: n_estimators= 200 at a max_depth=7
Validation Accuracy:.7947
 Passed- Accuracy of 81.6% exceeds the required threshold of .75


The updated model post additional testing confirms that this model is the strongest of the 3 models tested with a better than anticipated and required accuracy of 75% threshold at 81.6% threshold it shows that the model learned well and can predict customer behavior better than the other 2 models tested earlier. 